In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/My Drive/Visdrone Dataset/VisDrone2019-DET-train.zip" -d /content/visdrone_DET


Streaming output truncated to the last 5000 lines.
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000140.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000141.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000142.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000143.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000144.jpg  
  inflating: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000145.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000146.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000147.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000148.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000149.jpg  

In [ ]:
!unzip "/content/drive/My Drive/Visdrone Dataset/VisDrone2019-DET-val.zip" -d /content/visdrone_DET

Archive:  /content/drive/My Drive/Visdrone Dataset/VisDrone2019-DET-val.zip
   creating: /content/visdrone_DET/VisDrone2019-DET-val/
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/.DS_Store  
   creating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_02999_d_0000005.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_03499_d_0000006.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_03999_d_0000007.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_04527_d_0000008.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_05249_d_0000009.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_05499_d_0000010.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_05999_d_0000011.txt  
  inflating: /content/visdrone_DET/VisDrone2

In [ ]:
!unzip "/content/drive/My Drive/VisDrone2019-DET-test-dev.zip" -d /content/visdrone_DET


Archive:  /content/drive/My Drive/VisDrone2019-DET-test-dev.zip
   creating: /content/visdrone_DET/annotations/
  inflating: /content/visdrone_DET/annotations/0000006_00159_d_0000001.txt  
  inflating: /content/visdrone_DET/annotations/0000006_00611_d_0000002.txt  
  inflating: /content/visdrone_DET/annotations/0000006_01111_d_0000003.txt  
  inflating: /content/visdrone_DET/annotations/0000006_01275_d_0000004.txt  
  inflating: /content/visdrone_DET/annotations/0000006_01659_d_0000004.txt  
  inflating: /content/visdrone_DET/annotations/0000006_02138_d_0000006.txt  
  inflating: /content/visdrone_DET/annotations/0000006_02616_d_0000007.txt  
  inflating: /content/visdrone_DET/annotations/0000006_03636_d_0000009.txt  
  inflating: /content/visdrone_DET/annotations/0000006_04050_d_0000010.txt  
  inflating: /content/visdrone_DET/annotations/0000006_04309_d_0000011.txt  
  inflating: /content/visdrone_DET/annotations/0000006_05168_d_0000013.txt  
  inflating: /content/visdrone_DET/annota

In [ ]:

train_dir = '/content/visdrone_DET/VisDrone2019-DET-train'
val_dir = '/content/visdrone_DET/VisDrone2019-DET-val'



In [ ]:
import os
test_dir = os.makedirs('/content/visdrone_DET/VisDrone2019-DET-test-dev', exist_ok=True)

In [ ]:

print(os.listdir('/content/visdrone_DET/VisDrone2019-DET-test-dev'))


[]


In [ ]:
!mv '/content/visdrone_DET/images'  '/content/visdrone_DET/VisDrone2019-DET-test-dev'


In [ ]:
!mv '/content/visdrone_DET/annotations'  '/content/visdrone_DET/VisDrone2019-DET-test-dev'

In [ ]:
import os
import cv2
from PIL import Image
import pandas as pd

def visdrone_to_yolo(img_dir,ann_dir,yolo_ann_dir):
    os.makedirs(yolo_ann_dir, exist_ok=True)
    # loop that goes through annotation directory
    for ann_file in os.listdir(ann_dir):
        if not ann_file.endswith('.txt'):
            continue

        img_file = ann_file.replace('.txt','.jpg')
        img_path = os.path.join(img_dir, img_file)
        ann_path = os.path.join(ann_dir, ann_file)
        yolo_ann_path = os.path.join(yolo_ann_dir, ann_file)

        if not os.path.exists(img_path):
            print(f"Image does not exist for annotation: {ann_file}, skipping.")
            continue


        img = Image.open(img_path)
        w, h = img.size

        yolo_ann_lines = []
        with open(ann_path,'r')as file:
            for lines in file:
                parts = lines.strip().split(',')
                if len(parts) < 8:
                    continue
                x, y, width, height, score, category, trunc, occ = map(float, parts[:8])
                category = int(category)

                # skip ignored regions
                if category == 0:
                    continue

                yolo_cls = category - 1
                x_center =  (x + width/2 ) / w
                y_center =  (y + height/2 ) / h
                norm_width = width / w
                norm_height = height / h

                yolo_ann_lines.append(f"{yolo_cls} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}")

        if yolo_ann_lines:
            with open(yolo_ann_path, 'w') as file:
                file.write('\n'.join(yolo_ann_lines))
        else:
            print(f"Skipping empty label file: {ann_file}")



visdrone_to_yolo(
    img_dir= '/content/visdrone_DET/VisDrone2019-DET-test-dev/images',
    ann_dir= '/content/visdrone_DET/VisDrone2019-DET-test-dev/annotations',
    yolo_ann_dir= '/content/visdrone_DET/VisDrone2019-DET-test-dev/labels'
)





In [ ]:
pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
pip install git+https://github.com/ultralytics/ultralytics.git@main


  Cloning https://github.com/ultralytics/ultralytics.git (to revision main) to /tmp/pip-req-build-rh87j27k
  Running command git clone --filter=blob:none --quiet https://github.com/ultralytics/ultralytics.git /tmp/pip-req-build-rh87j27k
  Resolved https://github.com/ultralytics/ultralytics.git to commit 72abe071bb5151eb82f742c72c9c53c4ccede974
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os

os.makedirs('/content/dataset', exist_ok=True)

yaml_content = """
path: /content/visdrone_DET
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('/content/dataset/visdrone.yaml', 'w') as f:
    f.write(yaml_content)


In [ ]:
import os

def clean_label_dir(labels_dir, max_class_idx=9):     #Removes lines from label files where the class index exceeds max_class_idx.


     for fname in os.listdir(labels_dir):
        if not fname.endswith('.txt'):
            continue  # skip non-label files
        fpath = os.path.join(labels_dir, fname)
        with open(fpath, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            try:
                class_idx = int(line.split()[0])
                if class_idx <= max_class_idx:
                    new_lines.append(line)
            except Exception as e:
                print(f"Skipping line in {fname}: {line.strip()} (Error: {e})")
        with open(fpath, 'w') as f:
            f.writelines(new_lines)

test_labels_dir = '/content/visdrone_DET/VisDrone2019-DET-test-dev/labels'
clean_label_dir(test_labels_dir, max_class_idx=9)



In [ ]:
from ultralytics import YOLOWorld

model = YOLOWorld("/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt")

# Run validation on your labeled validation dataset
results_test = model.val(
    data='/content/dataset/visdrone.yaml' , split = 'test',
    augment = True,
    mosaic=1.0,
    conf=0.25,                          # Confidence threshold
    iou = 0.5,
    save=True,                          # Save results
    plots = True
)


Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-worldv2 summary (fused): 87 layers, 12,749,288 parameters, 0 gradients, 34.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2642.0±368.4 MB/s, size: 205.5 KB)


val: Scanning /content/visdrone_DET/VisDrone2019-DET-test-dev/labels.cache... 1610 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1610/1610 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 101/101 [00:38<00:00,  2.60it/s]


                   all       1610      75102      0.554      0.399      0.472      0.306
            pedestrian       1197      21006      0.642      0.338       0.51      0.251
                people        797       6376      0.634      0.158      0.391      0.184
               bicycle        377       1302      0.347      0.174      0.256      0.135
                   car       1530      28074      0.776      0.778      0.811      0.577
                   van       1168       5771      0.523      0.441      0.471      0.356
                 truck        750       2659      0.535      0.512      0.532      0.385
              tricycle        245        530      0.328      0.364      0.293      0.187
       awning-tricycle        233        599      0.423      0.262      0.308      0.212
                   bus        838       2940      0.765      0.548      0.677      0.538
                 motor        794       5845      0.569      0.413      0.471      0.236
Speed: 0.9ms preproce

In [ ]:

print("mAP50-95:", results_test.box.map)
print("mAP50:", results_test.box.map50)
print("Mean Precision:", results_test.box.mp)
print("Mean Recall:", results_test.box.mr)
print("Per-class mAP:", results_test.box.maps)


mAP50-95: 0.30622788821272184
mAP50: 0.4719557442424821
Mean Precision: 0.5540820160991557
Mean Recall: 0.39871597228292294
Per-class mAP: [    0.25102     0.18401     0.13512     0.57721     0.35637     0.38473     0.18715     0.21237     0.53804     0.23626]
